# 🤖 Baseline BERT Pipeline for AI vs Human Text Classification

이 노트북은 `klue/roberta-base` 모델을 사용하여 문단이 Human인지 AI인지 판별하는 베이스라인 모델을 학습합니다.

## 📋 프로세스 (Pipeline)
1. **환경 설정 (Setup)**: 라이브러리 임포트 및 시드 고정
2. **데이터 로드 (Data Load)**: `train.csv` 로드 및 Train/Validation 분리
3. **토크나이저 & 데이터셋 (Tokenizer & Dataset)**: `AutoTokenizer` 및 Custom Dataset 클래스 정의
4. **[검증] 데이터셋 확인 (Verification)**: 데이터 로더의 출력 차원(Dimension) 확인
5. **모델 정의 (Model)**: `AutoModelForSequenceClassification` 로드
6. **학습 루프 (Training Loop)**: Epoch 별 학습 및 검증 진행
7. **검증 및 저장 (Evaluation & Save)**: 검증 데이터셋 기준 성능 평가 및 모델 저장

In [1]:
# Colab 런타임에 파일 다운로드
# !pip install -q gdown
!gdown --id 1VDZetXszE7pLYjh4ku9GRm2DO7SHi8Ya -O /content/train.csv

DATA_PATH = '/content/drive/MyDrive/멋사_프로젝트_01/train.csv'  # 경로 수정

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1VDZetXszE7pLYjh4ku9GRm2DO7SHi8Ya
From (redirected): https://drive.google.com/uc?id=1VDZetXszE7pLYjh4ku9GRm2DO7SHi8Ya&confirm=t&uuid=59bda846-be4d-4c2a-a22c-97239e6fcc9a
To: /content/train.csv
100% 537M/537M [00:06<00:00, 89.1MB/s] 


In [7]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  # Transformers의 AdamW가 Deprecated 될 수 있어 torch.optim 사용 권장
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from tqdm import tqdm

# 시드 고정 (재현성 확보)
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

SEED = 42
seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"🔹 사용 디바이스: {device}")

🔹 사용 디바이스: cuda


## ⚙️ 설정 (Configuration)

In [11]:
MODEL_NAME = 'klue/roberta-base'  # 한국어 성능이 우수한 RoBERTa Base 모델
BATCH_SIZE = 32
MAX_LEN = 512  # 입력 토큰 최대 길이
EPOCHS = 3
LEARNING_RATE = 2e-5
DATA_PATH = '/content/train.csv'
SAVE_PATH = './best_model'

## 📂 데이터 로드 및 전처리 (Data Loading)

In [12]:
# 데이터 불러오기
df = pd.read_csv(DATA_PATH)
print(f"전체 데이터 개수: {len(df)}")

# Train / Validation 분리 (8:2)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['generated'])

print(f"Train 개수: {len(train_df)}, Validation 개수: {len(val_df)}")
display(train_df.head())

전체 데이터 개수: 97172
Train 개수: 77737, Validation 개수: 19435


,title,full_text,generated
43399,뉴욕의 역사,유럽에서 뉴욕을 처음으로 방문한 에스티방 고메스는 스페인 제국의 함선으로 1524년...,1
13509,코니시 카츠유키,용자왕 가오가이가의 볼포그 역으로 데뷔. 켄 프로덕션 소속. 카츠타 성우 학원 11...,0
19165,에어리어의 기사,"《에어리어의 기사》는 일본의 이가노 히로아키의 만화로, 소년 만화잡지 「주간 소년 ...",0
10518,기동전사 건담 역습의 샤아,"《기동전사 건담 역습의 샤아》(, )는 1988년 3월 12일에 개봉된 건담 시리즈...",0
25270,그리고리 쿨리크,"독소 전쟁이 발발한 후 독일군에게 참패를 당해 계급이 강등되었고, 제2차 세계대전 ...",0


## 🧩 토크나이저 로드 (Load Tokenizer)

In [13]:
# 토크나이저 로드 (use_fast=True 권장)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print(f"✅ Tokenizer 타입: {type(tokenizer).__name__}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


✅ Tokenizer 타입: BertTokenizerFast


## 🧩 Custom Dataset & DataLoader

In [14]:
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df['full_text'].values
        self.labels = df['generated'].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        # 토크나이저의 __call__ 메서드 사용 (encode_plus보다 안정적)
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        # input_ids: 토큰화된 단어들의 정수 ID
        # attention_mask: 실제 단어(1)와 패딩(0)을 구분하는 마스크
        # labels: 정답 라벨 (0: Human, 1: AI)
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 데이터셋 및 데이터로더 생성
train_dataset = TextDataset(train_df, tokenizer, MAX_LEN)
val_dataset = TextDataset(val_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

## ✅ [검증] 데이터 차원 및 파이프라인 확인 (Verification)
학습 시작 전, 데이터로더가 올바른 형태(Shape)의 텐서를 뱉어내는지 확인합니다.

In [7]:
# 첫 번째 배치를 가져와서 차원 확인
sample_batch = next(iter(train_loader))

input_ids_shape = sample_batch['input_ids'].shape
attention_mask_shape = sample_batch['attention_mask'].shape
labels_shape = sample_batch['labels'].shape

print(f"🔹 Batch Size: {BATCH_SIZE}")
print(f"🔹 Input IDs Shape: {input_ids_shape}  -> (Batch_Size, Max_Len)")
print(f"🔹 Attention Mask Shape: {attention_mask_shape} -> (Batch_Size, Max_Len)")
print(f"🔹 Labels Shape: {labels_shape} -> (Batch_Size)")

assert input_ids_shape == (BATCH_SIZE, MAX_LEN), "Input IDs 차원 오류"
assert labels_shape == (BATCH_SIZE,), "Labels 차원 오류"
print("✅ 데이터 파이프라인 차원 검증 완료")

🔹 Batch Size: 32
🔹 Input IDs Shape: torch.Size([32, 512])  -> (Batch_Size, Max_Len)
🔹 Attention Mask Shape: torch.Size([32, 512]) -> (Batch_Size, Max_Len)
🔹 Labels Shape: torch.Size([32]) -> (Batch_Size)
✅ 데이터 파이프라인 차원 검증 완료


## 🧠 모델 정의 (Model Definition)

In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # Human(0), AI(1)
)
model.to(device)

# 옵티마이저 설정 (torch.optim.AdamW 사용)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 🚀 학습 및 검증 (Training & Validation Loop)
변수 설명:
*   `outputs.logits`: 모델이 예측한 최종 점수 (Softmax 전 단계). Shape: `[Batch_Size, 2]`
*   `preds`: Logits 중 더 큰 값을 가진 Class Index (0 또는 1). Shape: `[Batch_Size]`

In [17]:
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples):
    model = model.train()
    losses = []
    correct_predictions = 0

    for d in tqdm(data_loader, desc="Training"):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=targets
        )

        loss = outputs.loss
        logits = outputs.logits
        
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / n_examples, np.mean(losses)

def eval_model(model, data_loader, loss_fn, device, n_examples):
    model = model.eval()
    losses = []
    correct_predictions = 0
    
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for d in tqdm(data_loader, desc="Evaluating"):
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=targets
            )

            loss = outputs.loss
            logits = outputs.logits
            
            _, preds = torch.max(logits, dim=1)
            probs = torch.nn.functional.softmax(logits, dim=1)[:, 1] # AI(1)일 확률

            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(targets.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    accuracy = correct_predictions.double() / n_examples
    auc_score = roc_auc_score(all_labels, all_probs)
    
    return accuracy, np.mean(losses), auc_score

In [ ]:
# 학습 실행
best_accuracy = 0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 10)

    train_acc, train_loss = train_epoch(
        model,
        train_loader,
        None,
        optimizer,
        device,
        scheduler,
        len(train_df)
    )

    val_acc, val_loss, val_auc = eval_model(
        model,
        val_loader,
        None,
        device,
        len(val_df)
    )

    print(f'Train loss {train_loss:.4f} accuracy {train_acc:.4f}')
    print(f'Val   loss {val_loss:.4f} accuracy {val_acc:.4f} AUC {val_auc:.4f}')

    # 모델 저장 (Best Accuracy 기준)
    if val_acc > best_accuracy:
        torch.save(model.state_dict(), 'best_model.pt')
        best_accuracy = val_acc
        print("✅ Best Model Saved!")

Epoch 1/3
----------


Evaluating: 100%|██████████| 608/608 [10:16<00:00,  1.01s/it]


Train loss 0.1420 accuracy 0.9651
Val   loss 0.1139 accuracy 0.9687 AUC 0.9443
✅ Best Model Saved!
Epoch 2/3
----------


Training:  30%|██▉       | 717/2430 [35:45<1:25:33,  3.00s/it]

: 

: 

Training:  30%|██▉       | 718/2430 [35:48<1:25:28,  3.00s/it]

In [ ]:
# 저장된 모델 로드 확인
model.load_state_dict(torch.load('best_model.pt'))
print("모델 로드 완료")

In [5]:
# 테스트 데이터 다운로드 (이미 했으면 스킵)
!gdown --id 15k3ZBne19_ASh_Uv8s8qutyEyXndlQHr -O /content/test.csv

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=15k3ZBne19_ASh_Uv8s8qutyEyXndlQHr
To: /content/test.csv
100% 1.44M/1.44M [00:00<00:00, 145MB/s]


In [18]:
# 테스트 데이터 로드
test_df = pd.read_csv('/content/test.csv')
print(f"Test 데이터 개수: {len(test_df)}")
display(test_df.head())

Test 데이터 개수: 1962


,ID,title,paragraph_index,paragraph_text
0,TEST_0000,공중 도덕의 의의와 필요성,0,도덕이란 원래 개인의 자각에서 출발해 자기 의지로써 행동하는 일이다. 그러므로 도덕...
1,TEST_0001,공중 도덕의 의의와 필요성,1,도덕은 단순히 개인의 문제나 사회의 문제로 한정될 수 없다. 개인적인 측면과 사회적...
2,TEST_0002,공중 도덕의 의의와 필요성,2,"여기에 이른바 공중도덕은 실천적, 사회적 도덕의 한 부문이다. 즉, 공중 도덕이라 ..."
3,TEST_0003,공중 도덕의 의의와 필요성,3,우리가 공동 생활을 하는 데 있어서 공중 도덕이 필요함은 위에서 말한 것처럼 알 수...
4,TEST_0004,풍습과 그 개선,0,인간 사회에서는 다 함께 지켜야 할 어떤 기준이 있어 이를 따르면 옳다고 하고 따르...


In [19]:
# 테스트용 Dataset (라벨 없음)
class TestDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df['paragraph_text'].values
        self.ids = df['ID'].values
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, item):
        text = str(self.texts[item])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'id': self.ids[item]
        }


In [20]:
# DataLoader
test_dataset = TestDataset(test_df, tokenizer, MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [21]:
# 최고 성능 모델 로드
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [26]:
# 추론
all_preds = []
all_ids = []
with torch.no_grad():
    for d in tqdm(test_loader, desc="Predicting"):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        _, preds = torch.max(outputs.logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_ids.extend(d['id'])
# 제출 파일 생성
submission = pd.DataFrame({
    'ID': all_ids,
    'generated': all_preds
})
submission.to_csv('submission.csv', index=False)
print("✅ submission.csv 저장 완료!")
display(submission.head())

Predicting: 100%|██████████| 62/62 [01:55<00:00,  1.86s/it]

✅ submission.csv 저장 완료!


,ID,generated
0,TEST_0000,0
1,TEST_0001,0
2,TEST_0002,0
3,TEST_0003,0
4,TEST_0004,0


In [32]:
from google.colab import files

# Colab 경로 확인 후 다운로드
import os
print(os.listdir('/content'))  # 파일 목록 확인

# submission.csv가 /content에 있다면:
files.download('/content/submission.csv')

['.config', 'best_model.pt', 'submission.csv', 'test.csv', 'train.csv', 'sample_data']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
# submission.csv 내용 출력
with open('submission.csv', 'r') as f:
    print(f.read())

ID,generated
TEST_0000,0
TEST_0001,0
TEST_0002,0
TEST_0003,0
TEST_0004,0
TEST_0005,0
TEST_0006,0
TEST_0007,0
TEST_0008,0
TEST_0009,0
TEST_0010,0
TEST_0011,0
TEST_0012,0
TEST_0013,0
TEST_0014,0
TEST_0015,0
TEST_0016,0
TEST_0017,0
TEST_0018,0
TEST_0019,0
TEST_0020,0
TEST_0021,0
TEST_0022,0
TEST_0023,0
TEST_0024,0
TEST_0025,0
TEST_0026,0
TEST_0027,0
TEST_0028,0
TEST_0029,0
TEST_0030,0
TEST_0031,0
TEST_0032,0
TEST_0033,0
TEST_0034,0
TEST_0035,0
TEST_0036,0
TEST_0037,0
TEST_0038,0
TEST_0039,0
TEST_0040,0
TEST_0041,0
TEST_0042,0
TEST_0043,0
TEST_0044,0
TEST_0045,0
TEST_0046,0
TEST_0047,0
TEST_0048,0
TEST_0049,1
TEST_0050,0
TEST_0051,1
TEST_0052,0
TEST_0053,1
TEST_0054,0
TEST_0055,0
TEST_0056,0
TEST_0057,0
TEST_0058,0
TEST_0059,0
TEST_0060,0
TEST_0061,0
TEST_0062,0
TEST_0063,0
TEST_0064,0
TEST_0065,0
TEST_0066,0
TEST_0067,0
TEST_0068,0
TEST_0069,0
TEST_0070,0
TEST_0071,0
TEST_0072,0
TEST_0073,0
TEST_0074,0
TEST_0075,0
TEST_0076,0
TEST_0077,0
TEST_0078,0
TEST_0079,0
TEST_0080,0
TEST_0081,0
TES